# 06 — The Complete Pipeline

This notebook uses **only the public API**: `MapEncryption`, `SchemeParams`, `SCHEME_VERSION`, and `_R_EARTH` (for the `metres_to_deg` helper). No internal functions are imported.

We encode all 250 cholera death locations from the 1854 Soho outbreak, render display positions, decode back to exact coordinates, and verify round-trip fidelity — all with interactive Plotly maps.

In [1]:
import secrets
import math
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from map_encryption import MapEncryption, SchemeParams, SCHEME_VERSION, _R_EARTH

def metres_to_deg(spread_m, at_lat):
    lat_deg = spread_m / 111_320
    lon_deg = spread_m / (111_320 * math.cos(math.radians(at_lat)))
    return lat_deg, lon_deg

# in production: load from secrets manager
MASTER_KEY = secrets.token_bytes(32)

CENTER_LAT, CENTER_LON = 51.513341, -0.136668  # Broadwick Street pump, Soho, London (1854 cholera outbreak)
enc = MapEncryption(MASTER_KEY, SchemeParams())

p = enc.params
print(f'Scheme version:    {SCHEME_VERSION}')
print(f'bin_size_m:        {p.bin_size_m} m')
print(f'jitter_max_frac:   {p.jitter_max_frac}')
print(f'prp_rounds:        {p.prp_rounds}')

Scheme version:    1
bin_size_m:        250 m
jitter_max_frac:   0.25
prp_rounds:        10


## Record Schema

| Field | Type | Size | Is it secret? | Purpose |
|-------|------|------|---------------|---------|
| `qxp` | int | 4–8 bytes | No — but opaque | Shuffled tile x-index; reveals nothing without PRP key |
| `qyp` | int | 4–8 bytes | No — but opaque | Shuffled tile y-index; reveals nothing without PRP key |
| `nonce` | bytes | 12 bytes | No — public | Unique per encode call; seeds AEAD and display jitter |
| `ct_resid` | bytes | 32 bytes | **Yes** | AEAD ciphertext of (rx, ry); holds full GPS precision |
| `tweak` | bytes | variable | No — public | Record-ID context; binds ciphertext to this record |
| `version` | int | 1 byte | No | Scheme version for forward-compatible migration |

In [2]:
import pandas as pd

deaths = pd.read_csv('data/cholera_deaths.csv')
n = len(deaths)  # 250 death locations recorded by Dr. John Snow

lats = deaths['LAT'].values
lons = deaths['LON'].values

records = [
    enc.encode(lats[i], lons[i],
               tweak=MapEncryption.make_tweak(record_id=int(deaths['FID'].iloc[i]),
                                              extra=b'nb06-demo'))
    for i in range(n)
]

render_pos = [enc.render_coordinates(r) for r in records]
print(f'Encoded {len(records)} cholera death locations (1854 Soho outbreak, Dr. John Snow).')
print(f'Sample record keys: {list(records[0].keys())}')


Encoded 250 cholera death locations (1854 Soho outbreak, Dr. John Snow).
Sample record keys: ['qxp', 'qyp', 'nonce', 'ct_resid', 'tweak', 'version']


In [3]:
decoded = [enc.decode(r) for r in records]
assert all(d is not None for d in decoded), 'decode returned None unexpectedly'

max_error = max(
    max(abs(decoded[i][0] - lats[i]), abs(decoded[i][1] - lons[i]))
    for i in range(n)
)
assert max_error < 1e-9, f'Round-trip error {max_error} exceeds tolerance'
print(f'Max round-trip error: {max_error:.2e} degrees — exact lossless encoding confirmed')

Max round-trip error: 3.55e-14 degrees — exact lossless encoding confirmed


In [4]:
import folium

# Map 1: Original vs Decoded — steelblue original, green decoded (smaller radius so overlaps are visible).
# Both point clouds are near Broadwick Street pump, Soho; they overlay exactly, confirming lossless round-trip.
# Display (render_coordinates) positions are in PRP-shuffled tile space — shown in the next cell.
m1 = folium.Map(location=[CENTER_LAT, CENTER_LON], zoom_start=12, tiles='cartodbpositron')
for i in range(n):
    folium.CircleMarker(
        location=[lats[i], lons[i]],
        radius=4,
        color='steelblue',
        fill=True, fill_color='steelblue', fill_opacity=0.6,
        tooltip='Original',
    ).add_to(m1)
    folium.CircleMarker(
        location=[decoded[i][0], decoded[i][1]],
        radius=2,
        color='green',
        fill=True, fill_color='green', fill_opacity=0.9,
        tooltip='Decoded',
    ).add_to(m1)
m1

In [5]:
import folium

# Display positions are in PRP-shuffled tile space — each record's tile was mapped to a
# pseudorandom location on the globe. Zoom level 2 reveals the intentional global scatter.
clat_r = sum(p[0] for p in render_pos) / n
clon_r = sum(p[1] for p in render_pos) / n
m2 = folium.Map(location=[clat_r, clon_r], zoom_start=2, tiles='cartodbpositron')
for rlat, rlon in render_pos:
    folium.CircleMarker(
        location=[rlat, rlon],
        radius=4,
        color='orange',
        fill=True, fill_color='orange', fill_opacity=0.6,
        tooltip='Display (render_coordinates)',
    ).add_to(m2)
m2

In [6]:
errors = [
    max(abs(decoded[i][0] - lats[i]), abs(decoded[i][1] - lons[i]))
    for i in range(n)
]

fig3 = px.histogram(
    x=errors,
    nbins=30,
    labels={'x': 'Max coordinate error (degrees)', 'y': 'Count'},
    title='Round-trip error distribution across 500 records',
    template='plotly_white',
    height=400
)
fig3.show()

print(f'All errors at or below: {max(errors):.2e} degrees')

All errors at or below: 3.55e-14 degrees


In [7]:
import struct as _struct

r0 = records[0]
wrong_enc = MapEncryption(secrets.token_bytes(32), SchemeParams())

scenarios = [
    ('Flip ct byte',
     lambda: enc.decode({**r0, 'ct_resid': bytes([r0['ct_resid'][0]^1]) + r0['ct_resid'][1:]})),
    ('Flip nonce byte',
     lambda: enc.decode({**r0, 'nonce': bytes([r0['nonce'][0]^1]) + r0['nonce'][1:]})),
    ('Shift qxp',
     lambda: enc.decode({**r0, 'qxp': r0['qxp'] + 1})),
    ('Wrong tweak',
     lambda: enc.decode({**r0, 'tweak': b'wrong-tweak'})),
    ('Wrong version',
     lambda: enc.decode({**r0, 'version': 999})),
    ('Wrong master key',
     lambda: wrong_enc.decode(r0)),
]

print(f'{"Scenario":<22} {"Result"}')
print('-' * 38)
all_none = True
for name, fn in scenarios:
    result = fn()
    status = 'PASS (None)' if result is None else 'FAIL'
    if result is not None:
        all_none = False
    print(f'  {name:<20} {status}')

assert all_none, 'At least one failure mode was not rejected'
print('\nAll 6 failure mode scenarios correctly returned None.')

Scenario               Result
--------------------------------------
  Flip ct byte         PASS (None)
  Flip nonce byte      PASS (None)
  Shift qxp            PASS (None)
  Wrong tweak          PASS (None)
  Wrong version        PASS (None)
  Wrong master key     PASS (None)

All 6 failure mode scenarios correctly returned None.


## Parameter Tuning

All participants in a deployment must use **identical** `SchemeParams`. A mismatch in any parameter will cause decode to fail silently (wrong tile recovery leads to wrong AD, so AEAD decryption returns None).

| Parameter | Default | Effect of increasing | Effect of decreasing |
|-----------|---------|----------------------|----------------------|
| `bin_size_m` | 250 m | Coarser spatial privacy; fewer total tiles | Finer grid; more tiles; less spatial masking |
| `jitter_max_frac` | 0.25 | Display pins spread further from tile centre | Display pins cluster near tile centre |
| `prp_rounds` | 10 | More cryptographic security for PRP; slower | Faster; below 4 rounds, security proofs weaken |

## References

- **Snow, J.** (1855). *On the Mode of Communication of Cholera* (2nd ed.). Churchill, London. — Source of the 1854 Soho cholera death and pump location dataset used throughout these notebooks.
- **Brody, H., Rip, M.R., Vinten-Johansen, P., Paneth, N., & Rachman, S.** (2000). Map-making and myth-making in Broad Street: the London cholera epidemic, 1854. *The Lancet, 356*(9223), 64–68.
- **Lin, Y.** (2023). Geo-indistinguishable masking: enhancing privacy protection in spatial point mapping. *Cartography and Geographic Information Science*. https://doi.org/10.1080/15230406.2023.2267967
- **Luby, M., & Rackoff, C.** (1988). How to construct pseudorandom permutations from pseudorandom functions. *SIAM Journal on Computing, 17*(2), 373–386. — Proof that ≥ 4 Feistel rounds produce a PRP.
- **Nir, Y., & Langley, A.** (2018). ChaCha20 and Poly1305 for IETF Protocols. RFC 8439. IETF. — Specification for the AEAD cipher used in this library.

## Glossary

| Term | Definition |
|------|-----------|
| **`MapEncryption`** | The public API class; wraps all four pipeline steps behind `encode()`, `decode()`, and `render_coordinates()`. |
| **`SchemeParams`** | A dataclass holding `bin_size_m`, `jitter_max_frac`, and `prp_rounds`; all participants in a deployment must use identical params or decryption fails silently. |
| **`SCHEME_VERSION`** | An integer stored in every record; `decode()` returns `None` for version mismatches, enabling forward-compatible key rotation. |
| **`make_tweak`** | A class method that packs a record ID and optional extra bytes into a canonical tweak; embeds `SCHEME_VERSION` to bind the ciphertext to its creation-time scheme. |
| **Round-trip fidelity** | The property that `decode(encode(lat, lon, tweak)) == (lat, lon)` to within floating-point precision; verified here to < 4 × 10⁻¹⁴ degrees for all 250 records. |
| **Failure mode** | A scenario in which `decode()` correctly returns `None` due to a tampered field, wrong key, wrong tweak, or version mismatch; demonstrates integrity protection. |
| **Parameter tuning** | Selecting `bin_size_m`, `jitter_max_frac`, and `prp_rounds` to balance spatial privacy, display utility, and performance for a given deployment. |
| **Display position** | The approximate (lat, lon) returned by `render_coordinates()`; located in pseudorandom tile space far from the true location; safe for public display. |